# NB09 — Pipeline unificado: 10 configuraciones × 9 cabezas

## Propósito

Entrenar todas las combinaciones (configuración de input × cabeza) bajo el **mismo protocolo experimental**:

- **StandardScaler** ajustado con datos de training de cada fold (sin leakage).
- **Grid search interno** mediante hold-out estratificado 80/20 dentro del training del fold.
- **Reentrenamiento con los mejores hiperparámetros** usando el training completo del fold.
- **Predicción** sobre validation fold (OOF) y sobre el test pool (ensemble entre folds).
- Misma semilla (`SEED=42`), mismo K=5, mismos splits.

Toda la lógica está en `tfm_pipeline_v2.py`. Este notebook orquesta las corridas y guarda predicciones.

## 10 configuraciones

Cinco inputs × dos poolings:

| Input | GAP+GMP (NB03) | Pool 2×2 (NB08) |
|---|---|---|
| E_A (estudio, 4 vistas) | 4096 dims | 16384 dims |
| E_B (estudio, 2 mapas asimetría) | 2048 dims | 8192 dims |
| E_AB (estudio, A+B) | 6144 dims | 24576 dims |
| M_A (mama, 2 vistas) | 2048 dims | 8192 dims |
| M_AB (mama, 2 vistas + 2 asimetrías) | 4096 dims | 16384 dims |

**M_AB** concatena para cada mama: `[CC propia, MLO propia, asym-CC del estudio, asym-MLO del estudio]`. La asimetría se repite para mama L y R del mismo estudio (es la misma información bilateral). Esto es razonable porque la asimetría no se puede asignar a una sola mama, pero sí puede dar contexto.

## 9 cabezas

LogReg L1, LogReg L2, LogReg ElasticNet, RandomForest, ExtraTrees, HistGradientBoosting, XGBoost, LightGBM, MLP.

## Total

10 × 9 = **90 corridas** (cada una son 5 folds + grid search interno + reentrenamiento). Guardado incremental tras cada corrida en `Outputs/Predicciones_v2/`. Si se interrumpe, al reanudar salta las ya hechas.

## Estimación de tiempo

Las corridas se ordenan de barata a cara: primero LogReg (segundos), luego boosting con early stopping (1-2 min), después árboles (1-2 min), finalmente MLP (1-3 min). Total estimado: **5-8 horas** en GPU RTX 4070 Super. Realista para dejar corriendo de noche.

In [1]:
import os, sys, time, json
import numpy as np
import pandas as pd
import torch

# Raíz del proyecto: por defecto, la carpeta padre de notebooks/.
# Sobrescribible con la variable de entorno TFM_PROJECT_ROOT.
BASE      = os.environ.get('TFM_PROJECT_ROOT',
                           os.path.abspath(os.path.join(os.getcwd(), '..')))
OUTPUTS      = os.path.join(BASE, 'outputs')
FEATURES_DIR = os.path.join(OUTPUTS, 'Features')
PRED_DIR_V2  = os.path.join(OUTPUTS, 'Predicciones_v2')
NB_DIR    = os.path.join(BASE, 'src')
os.makedirs(PRED_DIR_V2, exist_ok=True)

sys.path.insert(0, NB_DIR)
from tfm_pipeline import (  # noqa: E402  # type: ignore
    HEAD_NAMES, GRIDS, train_kfold_unified, SEED,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device: {DEVICE}')
print(f'Predicciones se guardarán en: {PRED_DIR_V2}')

Device: cuda
Predicciones se guardarán en: C:\Users\victo\Documents\TFM\Proyecto\Outputs\Predicciones_v2


## 1. Cargar features y metadata

Cargamos los dos sets de features (GAP+GMP del NB03, pool 2×2 del NB08) y el `metadata.csv` único.

In [2]:
X_view_gg = np.load(os.path.join(FEATURES_DIR, 'X_view.npy'))    # (N, 4, 1024) — GAP+GMP
X_asym_gg = np.load(os.path.join(FEATURES_DIR, 'X_asym.npy'))    # (N, 2, 1024) — GAP+GMP
X_view_22 = np.load(os.path.join(FEATURES_DIR, 'X_view_22.npy')) # (N, 4, 4096) — pool 2×2
X_asym_22 = np.load(os.path.join(FEATURES_DIR, 'X_asym_22.npy')) # (N, 2, 4096) — pool 2×2
meta      = pd.read_csv(os.path.join(FEATURES_DIR, 'metadata.csv'))

N = len(meta)
print(f'N estudios: {N}')
print(f'X_view_gg: {X_view_gg.shape}   X_view_22: {X_view_22.shape}')
print(f'X_asym_gg: {X_asym_gg.shape}   X_asym_22: {X_asym_22.shape}')

# Etiquetas y splits
y_E = meta['y_estudio'].values.astype(int)
split_E = meta['split'].values
is_train_E = (split_E == 'training')
is_test_E  = (split_E == 'test')

# Nivel mama: replicamos L y R
y_M = np.concatenate([meta['y_L'].values, meta['y_R'].values]).astype(int)
groups_M = np.concatenate([meta['study_id'].values, meta['study_id'].values])
split_M = np.concatenate([meta['split'].values, meta['split'].values])
is_train_M = (split_M == 'training')
is_test_M  = (split_M == 'test')

print(f'\nNivel estudio: train={is_train_E.sum()} (pos={int(y_E[is_train_E].sum())})  '
      f'test={is_test_E.sum()} (pos={int(y_E[is_test_E].sum())})')
print(f'Nivel mama:    train={is_train_M.sum()} (pos={int(y_M[is_train_M].sum())})  '
      f'test={is_test_M.sum()} (pos={int(y_M[is_test_M].sum())})')

N estudios: 4999
X_view_gg: (4999, 4, 1024)   X_view_22: (4999, 4, 4096)
X_asym_gg: (4999, 2, 1024)   X_asym_22: (4999, 2, 4096)

Nivel estudio: train=3999 (pos=385)  test=1000 (pos=96)
Nivel mama:    train=7998 (pos=395)  test=2000 (pos=99)


## 2. Construir las 10 configuraciones de input

Para cada configuración devolvemos: `X`, `level` ('estudio' o 'mama'), `groups` (None para estudio, study_id para mama).

In [3]:
def build_configs():
    """
    Devuelve dict {config_name: dict con X, y, level, is_train, is_test, groups}.
    Orden vistas en X_view: 0=L-CC, 1=R-CC, 2=L-MLO, 3=R-MLO
    Orden mapas asym:       0=asym-CC, 1=asym-MLO
    """
    configs = {}
    
    for tag, X_v, X_a in [('gg', X_view_gg, X_asym_gg), ('22', X_view_22, X_asym_22)]:
        # E_A: las 4 vistas aplanadas
        X_EA = X_v.reshape(N, -1)
        # E_B: los 2 mapas de asimetría aplanados
        X_EB = X_a.reshape(N, -1)
        # E_AB: E_A + E_B concatenados
        X_EAB = np.concatenate([X_EA, X_EB], axis=1)
        
        # M_A: mama L = [L-CC, L-MLO];  mama R = [R-CC, R-MLO]
        X_MA_L = np.concatenate([X_v[:, 0, :], X_v[:, 2, :]], axis=1)
        X_MA_R = np.concatenate([X_v[:, 1, :], X_v[:, 3, :]], axis=1)
        X_MA   = np.concatenate([X_MA_L, X_MA_R], axis=0)
        
        # M_AB: M_A + las 2 features de asimetría del estudio (repetidas para L y R)
        X_asym_flat = X_a.reshape(N, -1)  # (N, 2*d)
        X_MAB_L = np.concatenate([X_MA_L, X_asym_flat], axis=1)
        X_MAB_R = np.concatenate([X_MA_R, X_asym_flat], axis=1)
        X_MAB   = np.concatenate([X_MAB_L, X_MAB_R], axis=0)
        
        configs[f'E_A_{tag}']  = dict(X=X_EA.astype(np.float32),  y=y_E, level='estudio',
                                       is_train=is_train_E, is_test=is_test_E, groups=None)
        configs[f'E_B_{tag}']  = dict(X=X_EB.astype(np.float32),  y=y_E, level='estudio',
                                       is_train=is_train_E, is_test=is_test_E, groups=None)
        configs[f'E_AB_{tag}'] = dict(X=X_EAB.astype(np.float32), y=y_E, level='estudio',
                                       is_train=is_train_E, is_test=is_test_E, groups=None)
        configs[f'M_A_{tag}']  = dict(X=X_MA.astype(np.float32),  y=y_M, level='mama',
                                       is_train=is_train_M, is_test=is_test_M, groups=groups_M)
        configs[f'M_AB_{tag}'] = dict(X=X_MAB.astype(np.float32), y=y_M, level='mama',
                                       is_train=is_train_M, is_test=is_test_M, groups=groups_M)
    return configs

configs = build_configs()
print('Configuraciones construidas:')
for name, cfg in configs.items():
    print(f'  {name:14s}  X={cfg["X"].shape}  level={cfg["level"]}')

Configuraciones construidas:
  E_A_gg          X=(4999, 4096)  level=estudio
  E_B_gg          X=(4999, 2048)  level=estudio
  E_AB_gg         X=(4999, 6144)  level=estudio
  M_A_gg          X=(9998, 2048)  level=mama
  M_AB_gg         X=(9998, 4096)  level=mama
  E_A_22          X=(4999, 16384)  level=estudio
  E_B_22          X=(4999, 8192)  level=estudio
  E_AB_22         X=(4999, 24576)  level=estudio
  M_A_22          X=(9998, 8192)  level=mama
  M_AB_22         X=(9998, 16384)  level=mama


## 3. Orden de las corridas

Ordenamos por coste estimado: cabezas baratas primero, MLP al final. Para configuraciones, primero las que sabemos que pueden ser buenas (M_A_22, M_AB_22) y luego el resto.

Cada par (config, head) se ejecuta una vez. Si el archivo de predicciones ya existe en disco, se salta (reanudación).

In [4]:
# Cabezas en orden ascendente de coste
HEAD_ORDER = ['logreg_l1', 'logreg_l2',
              'histgb', 'lgbm', 'xgb',
              'rf', 'extratrees', 'mlp']

# Configuraciones en orden de potencial interés (mama 2×2 primero por evidencia previa)
CONFIG_ORDER = [
    'M_A_22',  'M_AB_22',
    'M_A_gg',  'M_AB_gg',
    'E_A_22',  'E_B_22',  'E_AB_22',
    'E_A_gg',  'E_B_gg',  'E_AB_gg',
]

total_runs = len(CONFIG_ORDER) * len(HEAD_ORDER)
print(f'Total corridas planeadas: {total_runs}')
print(f'Orden de cabezas: {HEAD_ORDER}')
print(f'Orden de configuraciones: {CONFIG_ORDER}')

Total corridas planeadas: 80
Orden de cabezas: ['logreg_l1', 'logreg_l2', 'histgb', 'lgbm', 'xgb', 'rf', 'extratrees', 'mlp']
Orden de configuraciones: ['M_A_22', 'M_AB_22', 'M_A_gg', 'M_AB_gg', 'E_A_22', 'E_B_22', 'E_AB_22', 'E_A_gg', 'E_B_gg', 'E_AB_gg']


## 4. Bucle principal con reanudación

Por cada (configuración, cabeza) entrena, evalúa y guarda predicciones OOF y test. Si ya existe el archivo, salta.

Salida en disco por cada corrida:
- `Predicciones_v2/{config}__{head}_oof.npy`
- `Predicciones_v2/{config}__{head}_test.npy`
- `Predicciones_v2/{config}__{head}_meta.json` (AUCs por fold, hparams elegidos)

Y al final, un resumen consolidado en `resumen_v2.csv`.

In [5]:
def run_pipeline(config_order=CONFIG_ORDER, head_order=HEAD_ORDER, skip_existing=True):
    """Ejecuta todas las corridas en orden, con reanudación si ya existen las predicciones."""
    n_done = 0
    n_skipped = 0
    n_failed = 0
    t_global = time.time()
    
    for cfg_name in config_order:
        cfg = configs[cfg_name]
        print(f'\n{"="*70}')
        print(f'CONFIGURACIÓN: {cfg_name}   ({cfg["X"].shape[1]} dims, level={cfg["level"]})')
        print("="*70)
        for head_name in head_order:
            run_id = f'{cfg_name}__{head_name}'
            oof_path  = os.path.join(PRED_DIR_V2, f'{run_id}_oof.npy')
            test_path = os.path.join(PRED_DIR_V2, f'{run_id}_test.npy')
            meta_path = os.path.join(PRED_DIR_V2, f'{run_id}_meta.json')
            
            if skip_existing and os.path.isfile(oof_path) and os.path.isfile(test_path):
                n_skipped += 1
                continue
            
            print(f'\n[{n_done+n_skipped+n_failed+1}/{total_runs}] {run_id}')
            t0 = time.time()
            try:
                result = train_kfold_unified(
                    head_name, cfg['X'], cfg['y'],
                    cfg['is_train'], cfg['is_test'],
                    groups=cfg['groups'], n_splits=5,
                    mlp_device=DEVICE, verbose=True,
                )
                np.save(oof_path,  result['oof_preds'])
                np.save(test_path, result['test_preds'])
                meta_info = {
                    'config': cfg_name, 'head': head_name,
                    'level': cfg['level'],
                    'oof_auc': float(result['oof_auc']),
                    'test_auc': float(result['test_auc']),
                    'mean_fold_auc': float(result['mean_fold_auc']),
                    'std_fold_auc':  float(result['std_fold_auc']),
                    'fold_aucs': [float(a) for a in result['fold_aucs']],
                    'fold_hps':  [dict(h) if h is not None else None for h in result['fold_hps']],
                    'elapsed_sec': time.time() - t0,
                }
                with open(meta_path, 'w') as f:
                    json.dump(meta_info, f, indent=2)
                n_done += 1
                print(f'  ✓ {run_id}  test_AUC={result["test_auc"]:.4f}  ({time.time()-t0:.1f}s)')
            except Exception as e:
                n_failed += 1
                print(f'  ✗ {run_id}  ERROR: {e}')
                with open(meta_path.replace('.json', '_ERROR.txt'), 'w') as f:
                    f.write(f'{type(e).__name__}: {e}\n')
        
        # Ahorro de memoria entre configuraciones
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    
    elapsed = time.time() - t_global
    print(f'\n{"="*70}')
    print(f'Pipeline completado. Tiempo total: {elapsed/60:.1f} min')
    print(f'  Ejecutadas: {n_done}    Saltadas (ya existían): {n_skipped}    Fallidas: {n_failed}')
    return n_done, n_skipped, n_failed

print('Función run_pipeline definida. Llama a run_pipeline() para empezar.')

Función run_pipeline definida. Llama a run_pipeline() para empezar.


## 5. Ejecutar

Esto es el cuerpo del entrenamiento. Tarda varias horas. Si se interrumpe (Ctrl+C, cierre, error), al volver a ejecutar reanuda desde donde se quedó (salta las corridas cuyos archivos ya existen).

In [6]:
n_done, n_skipped, n_failed = run_pipeline()


CONFIGURACIÓN: M_A_22   (8192 dims, level=mama)

CONFIGURACIÓN: M_AB_22   (16384 dims, level=mama)

CONFIGURACIÓN: M_A_gg   (2048 dims, level=mama)

CONFIGURACIÓN: M_AB_gg   (4096 dims, level=mama)

CONFIGURACIÓN: E_A_22   (16384 dims, level=estudio)

[33/80] E_A_22__logreg_l1
    fold 1: val_AUC=0.6235  hp={'C': 0.1}  (207.1s)
    fold 2: val_AUC=0.6067  hp={'C': 0.1}  (207.6s)
    fold 3: val_AUC=0.6276  hp={'C': 0.1}  (212.2s)
    fold 4: val_AUC=0.7197  hp={'C': 1.0}  (234.8s)
    fold 5: val_AUC=0.6209  hp={'C': 0.1}  (209.7s)
    → fold AUCs: ['0.6235', '0.6067', '0.6276', '0.7197', '0.6209']
    → mean fold AUC: 0.6397  (std 0.0406)
    → OOF AUC: 0.6400     Test AUC: 0.6122
  ✓ E_A_22__logreg_l1  test_AUC=0.6122  (1071.4s)

[34/80] E_A_22__logreg_l2
    fold 1: val_AUC=0.6103  hp={'C': 10.0}  (18.6s)
    fold 2: val_AUC=0.5995  hp={'C': 0.1}  (23.4s)
    fold 3: val_AUC=0.5849  hp={'C': 0.1}  (21.8s)
    fold 4: val_AUC=0.6070  hp={'C': 10.0}  (22.8s)
    fold 5: val_AUC=0.558

## 6. Consolidar resultados en un único CSV

Lee todos los `*_meta.json` y produce una tabla con AUC OOF, AUC test, mean/std fold por (configuración, cabeza). Esta tabla es la base para el NB10 (evaluación detallada con IC bootstrap, DeLong, Brier, ECE) y para la memoria.

In [7]:
import glob

rows = []
for meta_path in sorted(glob.glob(os.path.join(PRED_DIR_V2, '*_meta.json'))):
    with open(meta_path) as f:
        info = json.load(f)
    rows.append({
        'config':         info['config'],
        'head':           info['head'],
        'level':          info['level'],
        'oof_auc':        info['oof_auc'],
        'mean_fold_auc':  info['mean_fold_auc'],
        'std_fold_auc':   info['std_fold_auc'],
        'test_auc':       info['test_auc'],
        'elapsed_sec':    info.get('elapsed_sec', np.nan),
    })

df_res = pd.DataFrame(rows)
df_res = df_res.sort_values('test_auc', ascending=False).reset_index(drop=True)

out_csv = os.path.join(PRED_DIR_V2, 'resumen_v2.csv')
df_res.to_csv(out_csv, index=False)
print(f'Guardado: {out_csv}')
print(f'Total filas: {len(df_res)}')
print()
print('TOP 10 por AUC test:')
print(df_res.head(10).to_string(index=False, float_format=lambda x: f'{x:.4f}'))

Guardado: C:\Users\victo\Documents\TFM\Proyecto\Outputs\Predicciones_v2\resumen_v2.csv
Total filas: 84

TOP 10 por AUC test:
 config      head level  oof_auc  mean_fold_auc  std_fold_auc  test_auc  elapsed_sec
 M_A_gg       mlp  mama   0.6442         0.6421        0.0333    0.6866      28.9059
 M_A_22       xgb  mama   0.6245         0.6527        0.0270    0.6813     402.9795
M_AB_gg    histgb  mama   0.6254         0.6359        0.0295    0.6811     181.2271
 M_A_22        rf  mama   0.6517         0.6575        0.0242    0.6803      95.4537
 M_A_gg logreg_en  mama   0.5928         0.5925        0.0460    0.6763     843.8041
M_AB_gg      lgbm  mama   0.6134         0.6107        0.0289    0.6758     108.2348
 M_A_gg logreg_l1  mama   0.5877         0.5887        0.0474    0.6729     516.4520
M_AB_gg       mlp  mama   0.6549         0.6561        0.0333    0.6709      37.4611
 M_A_gg    histgb  mama   0.6224         0.6279        0.0394    0.6692      85.5656
 M_A_gg        rf  mama  

## 7. Vista por configuración

Pivot table que muestra el mejor AUC por (configuración, cabeza) para identificar patrones.

In [8]:
pivot = df_res.pivot(index='config', columns='head', values='test_auc')
# Orden de filas: igual que CONFIG_ORDER (mejor primero)
pivot = pivot.reindex(CONFIG_ORDER)
# Orden de columnas: igual que HEAD_ORDER
pivot = pivot[HEAD_ORDER]
print('AUC test por (configuración × cabeza):')
print(pivot.to_string(float_format=lambda x: f'{x:.4f}'))

print('\nMejor cabeza por configuración:')
for cfg in CONFIG_ORDER:
    if cfg in pivot.index:
        best_head = pivot.loc[cfg].idxmax()
        best_auc  = pivot.loc[cfg].max()
        print(f'  {cfg:14s}  best: {best_head:12s}  AUC={best_auc:.4f}')

print('\nMejor configuración por cabeza:')
for head in HEAD_ORDER:
    if head in pivot.columns:
        best_cfg = pivot[head].idxmax()
        best_auc = pivot[head].max()
        print(f'  {head:12s}  best: {best_cfg:14s}  AUC={best_auc:.4f}')

AUC test por (configuración × cabeza):
head     logreg_l1  logreg_l2  histgb   lgbm    xgb     rf  extratrees    mlp
config                                                                       
M_A_22      0.6340     0.6158  0.6653 0.6415 0.6813 0.6803      0.6347 0.6422
M_AB_22     0.6217     0.6185  0.6580 0.6088 0.6330 0.6458      0.6173 0.6386
M_A_gg      0.6729     0.6643  0.6692 0.6477 0.6373 0.6661      0.6567 0.6866
M_AB_gg     0.6458     0.6443  0.6811 0.6758 0.6438 0.6472      0.6454 0.6709
E_A_22      0.6122     0.5828  0.6347 0.6126 0.6315 0.6374      0.6363 0.5948
E_B_22      0.5891     0.5885  0.5772 0.5798 0.6120 0.5813      0.6058 0.5964
E_AB_22     0.5977     0.5863  0.6247 0.6168 0.6429 0.6278      0.6266 0.5920
E_A_gg      0.6465     0.6075  0.6061 0.5779 0.6231 0.6499      0.6318 0.6237
E_B_gg      0.6117     0.5917  0.6177 0.6095 0.6139 0.6073      0.6260 0.6324
E_AB_gg     0.6437     0.6005  0.6393 0.6283 0.6258 0.6251      0.6270 0.6270

Mejor cabeza por configu

## Siguiente paso

`NB10 — Evaluación detallada` calculará para la mejor combinación (y para todas las del top):
- AUC + IC 95% por bootstrap (1.000 remuestreos).
- Average Precision (PR-AUC).
- Brier score y ECE (calibración).
- Test pareado de DeLong para comparaciones entre configuraciones y cabezas.
- Análisis estratificado por densidad mamaria.
- Comparación contra los resultados del Hito 2 (NB05/NB07) para cuantificar la mejora.

`NB11 — Fusión con densidad` reentrenará la mejor configuración añadiendo densidad como covariable, igual que en NB07 pero ahora sobre la mejor cabeza/pooling encontrada.

`NB12 — Calibración post-hoc` aplicará Platt e isotónica sobre la mejor MLP para mejorar Brier y ECE.